# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [02:17<00:00, 27.40s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: 6-Foot Artificial Ficus Tree for $36 + free shipping\nDetails: Buy Now at Walmart\nFeatures: \nURL: https://www.dealnews.com/6-Foot-Artificial-Ficus-Tree-for-36-free-shipping/21764185.html?iref=rss-c196'

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Tronsmart - Bang Max Portable Bluetooth Speaker for $180 + free shipping
Details: That's a savings of $50. Buy Now at Best Buy
Features: 130W Max Booming Bass Deep Bass and SoundPulse Technology Beat-Driven Light Show IPX6 Waterproof and Solid Materials Listen All-Day Anywhere
URL: https://www.dealnews.com/Tronsmart-Bang-Max-Portable-Bluetooth-Speaker-for-180-free-s

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
len(result.deals)

5

In [12]:
result.deals[1]

Deal(product_description='The Refurbished Apple MacBook Air M1 (2020) features a powerful Apple M1 chip with an 8-Core CPU, ensuring fast performance for handling tasks seamlessly. Its 13.3-inch Retina display boasts a resolution of 2560x1600, offering vibrant colors and sharp text. The laptop includes 8GB of RAM and a 128GB SSD, making it perfect for everyday use, whether for work, school, or entertainment. With macOS Big Sur, it provides an intuitive user experience, making this a top choice for Apple enthusiasts.', price=400.0, url='https://www.dealnews.com/products/Apple/Apple-Mac-Book-Air-M1-13-Laptop-2020/165091.html?iref=rss-c39')

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent()
result = agent.scan()

In [15]:
result

DealSelection(deals=[Deal(product_description="The Tronsmart Bang Max is a top-tier portable Bluetooth speaker that delivers an impressive 130W of booming bass thanks to its advanced SoundPulse Technology. Featuring a vibrant beat-driven light show and a rugged IPX6 waterproof rating, this speaker is perfect for outdoor parties or home use. With solid materials ensuring durability, you'll enjoy an all-day listening experience anywhere you go.", price=180.0, url='https://www.dealnews.com/Tronsmart-Bang-Max-Portable-Bluetooth-Speaker-for-180-free-shipping/21764205.html?iref=rss-c142'), Deal(product_description="The Anker Solix C300X is a versatile 288Wh portable power station equipped with a 60W solar panel, allowing you to harness solar energy on the go. This power station features a robust 300W output and includes seven outlets, including two rapid charging USB-C ports. Whether you're camping, attending a tailgate, or facing a power outage, this compact and powerful device ensures you 